# BERT Token Classification: POS Tagging & Chunking

## Install Libraries

In [ ]:
!pip install transformers datasets seqeval

## Import Libraries

In [ ]:

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import classification_report, f1_score


## Task 1: Load Dataset (CoNLL-2003 for Chunking)

In [ ]:

dataset = load_dataset("conll2003")
label_list = dataset["train"].features["ner_tags"].feature.names
print(label_list)


## Task 2: Tokenization & Label Alignment

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)


## Task 3: Model Setup

In [ ]:

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list)
)


## Task 4: Training

In [ ]:

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer
)

trainer.train()


## Task 5: Evaluation

In [ ]:

predictions, labels, _ = trainer.predict(tokenized_dataset["validation"])
preds = np.argmax(predictions, axis=2)

true_labels = [[label_list[l] for l in label if l != -100] for label in labels]
true_preds = [[label_list[p] for (p, l) in zip(pred, label) if l != -100] for pred, label in zip(preds, labels)]

print(classification_report(true_labels, true_preds))


## Task 6: Inference

In [ ]:

sentence = "John works at Google in California"
inputs = tokenizer(sentence.split(), return_tensors="pt", is_split_into_words=True)

outputs = model(**inputs).logits
predictions = outputs.argmax(dim=2)

tokens = sentence.split()
predicted_labels = [label_list[p.item()] for p in predictions[0]]

for token, label in zip(tokens, predicted_labels):
    print(token, "->", label)


## Task 7: Comparison


POS Tagging → Word-level grammar tagging  
Chunking → Phrase-level grouping  
Chunking is more complex than POS tagging.


## Task 8: Report


Challenges:
- Handling subword tokens
- Label alignment
- Training time

Insights:
- Transformer models give high accuracy
- Proper preprocessing is critical
